In [1]:
!pip install pandas
!pip install dash
!pip install plotly

In [2]:
!pip install js

In [3]:
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

In [4]:
#from js import fetch
import io

In [5]:
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv')
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()


In [6]:
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [7]:
# create a dash application
app = dash.Dash(__name__)
app.config.suppress_callback_exceptions = True

In [8]:

# create an app layout
app.layout = html.Div([
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36','font-size': 40}
   ),

    html.Div([
        dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'},
            *[{'label': site, 'value': site} for site in spacex_df['Launch Site'].unique()]
        ],
        value='ALL',
        placeholder='Select a Launch Site',
        searchable=True
        )
    ]),
    html.Br(),

    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),
        
    html.P("payload range (kg):"),
    html.Div([
        dcc.RangeSlider(
            id ='payload-slider',
            min = 0,
            max = 10000,
            step = 1000,
            marks = {i: f'{i}' for i in range(0, 10001, 2500)},
            value = [min_payload, max_payload]
        )
    ]),
    html.Br(),

    html.Div(dcc.Graph(id='success-payload-scatter-chart'))   
])
                   

In [9]:
# callback for pie chart
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        df = spacex_df[spacex_df['class'] == 1]
        fig = px.pie(df, names='Launch Site', title='Total Successful Launches by Site')
    else:
        df = spacex_df[spacex_df['Launch Site'] == selected_site]
        fig = px.pie(df, names='class', title=f'Success vs. Failure for {selected_site}')
    return fig

In [10]:
# callback for scatter plot
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [
        Input(component_id='site-dropdown', component_property='value'),
        Input(component_id='payload-slider', component_property='value')
    ]
)
def update_scatter(selected_site, payload_range):
    low, high = payload_range
    df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= low) & 
        (spacex_df['Payload Mass (kg)'] <= high)
    ]
    
    if selected_site != 'ALL':
        df = df[df['Launch Site'] == selected_site]
    
    fig = px.scatter(
        df,
        x='Payload Mass (kg)',
        y='class',
        color='Booster Version Category',
        title=f'Payload vs. Success for {selected_site}'
    )
    return fig


In [11]:
if __name__ == '__main__':
    app.run(debug=True)